# 2D Pose Evaluation Notebook

This notebook evaluates the performance of various 2D pose estimation models against ground truth data.

**Steps:**
1. Load ground truth annotations from the `labeled_poses` folder.
2. Detect persons using the YOLOv11l model.
3. Run the following pose estimation models:
   - Sapiens-1B
   - RTMPose-l Wholebody
   - Additional models specified in a configuration array.
4. Use the RTMPose Hand model on cropped hand regions from the wholebody pose model.
5. Evaluate predictions against ground truth using COCO metrics (AP and AR).

In [7]:
# Import Required Libraries
import os
import json
import cv2
import numpy as np
from tqdm.notebook import tqdm
from mmpose.apis import init_model, inference_topdown
from mmpose.evaluation.functional import nms
from helpers.json_handling import read_keypoints_mmpose

# Import visualization and helper functions from predictors.py

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

print("Libraries and helpers imported successfully.")

Libraries and helpers imported successfully.


In [8]:
# Configuration

# Paths
dataset = 'cha'  # dataset name
data_collections = [f'{dataset}/cha1', f'{dataset}/cha2', f'{dataset}/cha3', f'{dataset}/cha4', f'{dataset}/cha5', f'{dataset}/cha6', f'{dataset}/cha7']  # dataset collection name
# Path to labeled 2D hand poses (ground truth)
labeled_frames = list(range(0, 300, 30))
input_base_dir = f'inputs/'  # Directory containing images
cam_names = [f'gopro{i}' for i in range(5, 13)]

method_overview = {
    'Sapiens': 'predictions_2d/sapiens',
    'RTMPose': 'predictions_2d/rtmpose',
    'DWPose': 'predictions_2d/dwpose',
    'RefinedPoses': 'predictions_2d/refined_poses',
    'RTMPoseHands': 'predictions_2d/rtmpose_hands',
    'DWPoseHands': 'predictions_2d/dwpose_hands',
    'SapiensHands': 'predictions_2d/sapiens_hands'
}

original_resolution = (3840, 2160)
DEVICE = 'cuda:0'

In [9]:
def load_labeled_2d_poses(data_collection, labeled_frames):
    # Path to labeled 2D hand poses (ground truth)
    labeled_2d_path = f'labeled_poses/{data_collection}/hand_poses_2d_labeled.npz'  # Update as needed
    # Load 2D Labeled Hand Poses
    data_2d = np.load(labeled_2d_path, allow_pickle=True)
    # Assume the file contains a dict: {cam_name: {frame_idx: {obj_id: {'keypoints': ndarray, 'scores': ndarray}}}}
    unindexed_labeled_2d_poses = data_2d['poses_2d'].item() if 'poses_2d' in data_2d else data_2d[list(data_2d.keys())[0]].item()

    cam_names = list(unindexed_labeled_2d_poses.keys())
    labeled_2d_poses = {k: {cam_name: {} for cam_name in cam_names} for k in labeled_frames}
    for cam_name in cam_names:
        for frame_idx in labeled_frames:
            for obj_id in unindexed_labeled_2d_poses[cam_name][frame_idx].keys():
                keypoints = unindexed_labeled_2d_poses[cam_name][frame_idx][obj_id]['keypoints']
                scores = unindexed_labeled_2d_poses[cam_name][frame_idx][obj_id]['keypoint_scores']
                # Ensure keypoints and scores are numpy arrays
                labeled_2d_poses[frame_idx][cam_name][obj_id] = {'keypoints':np.array(keypoints), 'scores': np.array(scores)}
    # print(f"Loaded labeled 2D hand poses for {len(labeled_2d_poses)} frames.")

    return labeled_2d_poses

In [10]:
def load_predictions(data_collection, base_path):
    pose_output_path = f'{base_path}/{data_collection}'
    # Read 2D keypoints from JSON files
    _, poses_2d_hands = read_keypoints_mmpose(
        pose_output_path, 
        cam_names, 
        num_frames=10,  # Use dynamically determined frame count
        num_body_keypoints=17,  # Number of keypoints for body
        num_hand_keypoints=21,  # Number of keypoints for hands
        suffix='_synced_cut',
        use_wholebody=True
    )
    return poses_2d_hands

In [11]:
poses_2d = {}
for data_collection in data_collections:
    poses_2d[data_collection] = {}
    for method, pose_path in method_overview.items():
        poses_2d[data_collection][method] = load_predictions(data_collection, pose_path)

print(len(poses_2d['cha/cha5']['Sapiens'][5]))  # Example access

body keypoints could not be accessed for frame 2
list index out of range
body keypoints could not be accessed for frame 3
list index out of range
body keypoints could not be accessed for frame 4
list index out of range
body keypoints could not be accessed for frame 5
list index out of range
body keypoints could not be accessed for frame 6
list index out of range
body keypoints could not be accessed for frame 8
list index out of range
body keypoints could not be accessed for frame 9
list index out of range
body keypoints could not be accessed for frame 3
list index out of range
body keypoints could not be accessed for frame 4
list index out of range
body keypoints could not be accessed for frame 1
list index out of range
body keypoints could not be accessed for frame 3
list index out of range
body keypoints could not be accessed for frame 3
list index out of range
body keypoints could not be accessed for frame 5
list index out of range
body keypoints could not be accessed for frame 6
li

In [12]:
import torch

def keypoint_similarity(gt_kpts, pred_kpts, sigmas, areas):
    """
    Params:
        gts_kpts: Ground-truth keypoints, Shape: [M, #kpts, 3],
                  where, M is the # of ground truth instances,
                         3 in the last dimension denotes coordinates: x,y, and visibility flag
                          
        pred_kpts: Prediction keypoints, Shape: [N, #kpts, 3]
                   where  N is the # of predicted instances,
 
        areas: Represent ground truth areas of shape: [M,]
 
    Returns:
        oks: The Object Keypoint Similarity (OKS) score tensor of shape: [M, N]
    """

    gt_kpts = torch.tensor(gt_kpts, dtype=torch.float32)
    pred_kpts = torch.tensor(pred_kpts, dtype=torch.float32)

    gt_kpts = gt_kpts.reshape(-1, 21, 3)  # [M, 21, 3]
    pred_kpts = pred_kpts.reshape(-1, 21, 3)  # [N, 21, 3]


    # epsilon to take care of div by 0 exception.
    EPSILON = torch.finfo(torch.float32).eps
     
    # Eucleidian dist squared:
    # d^2 = (x1 - x2)^2 + (y1 - y2)^2
    # Shape: (M, N, #kpts) --> [M, N, 17]
    dist_sq = (gt_kpts[:,None,:,0] - pred_kpts[...,0])**2 + (gt_kpts[:,None,:,1] - pred_kpts[...,1])**2
 
    # Boolean ground-truth visibility mask for v_i > 0. Shape: [M, #kpts] --> [M, 17]
    vis_mask = gt_kpts[..., 2].int() > 0
 
    # COCO assigns k = 2σ.
    k = 2*sigmas
 
    # Denominator in the exponent term. Shape: [M, 1, #kpts] --> [M, 1, 17]
    denom = 2 * (k**2) * (areas[:,None, None] + EPSILON)
 
    # Exponent term. Shape: [M, N, #kpts] --> [M, N, 17]
    exp_term = dist_sq / denom
    
    # Object Keypoint Similarity. Shape: (M, N)
    oks = (torch.exp(-exp_term) * vis_mask[:, None, :]).sum(-1) / (vis_mask[:, None, :].sum(-1) + EPSILON).diag()

    return oks

def compute_areas(keypoints):
    """
    Compute the areas of the keypoint instances.
    """
    # Areas are computed as the bounding box area for each instance
    areas = []
    for kpts in keypoints:
        x_coords = kpts[:, 0]
        y_coords = kpts[:, 1]
        if len(x_coords) > 0 and len(y_coords) > 0:
            x_min = x_coords.min()
            x_max = x_coords.max()
            y_min = y_coords.min()
            y_max = y_coords.max()
            area = (x_max - x_min) * (y_max - y_min)
            areas.append(area)
        else:
            areas.append(0)
    return np.array(areas)

def compute_average_precision(oks):
    """
    Compute Average Precision (AP) from OKS scores.
    """
    thresholds = np.linspace(0.5, 0.95, 10)
    ap = 0
    for t in thresholds:
        tp = (oks >= t).sum()
        fp = (oks < t).sum()
        precision = tp / (tp + fp + 1e-6)
        ap += precision
    ap /= len(thresholds)
    return ap

def compute_mean_joint_error(gt_kpts, pred_kpts):
    """
    Compute Mean Joint Error (MJE) between ground truth and predicted keypoints.
    """
    masking = gt_kpts[:, :, 2] > 0
    masking = masking & (pred_kpts[:, :, 2] > 0)

    gt_kpts = gt_kpts[masking]
    pred_kpts = pred_kpts[masking]
    return np.mean(np.linalg.norm(gt_kpts[:, :2] - pred_kpts[:, :2], axis=-1))

def compute_pck(gt_kpts, pred_kpts):
    """
    Compute Percentage of Correct Keypoints (PCK) between ground truth and predicted keypoints.
    Use pixel thresholds of [5, 10, 20, 30] # in pixels
    """
    pck = np.zeros((gt_kpts.shape[0], gt_kpts.shape[1], 4))
    thresholds = [5, 10, 20, 30]
    for i in range(gt_kpts.shape[0]):
        for j in range(gt_kpts.shape[1]):
            if gt_kpts[i, j, 2] > 0:  # If the ground truth keypoint is visible
                dist = np.linalg.norm(gt_kpts[i, j, :2] - pred_kpts[i, j, :2])
                for k, thr in enumerate(thresholds):
                    pck[i, j, k] = float(dist < thr)
                

    return np.mean(pck)

def evaluate_keypoint_predictions(labeled_2d_poses, method_predictions):
    """
    Evaluate keypoint predictions using OKS and other metrics.
    """
    gt_kpts = []
    pred_kpts = []
    for idx, frame_idx in enumerate(labeled_frames):
        for cam_idx, cam in enumerate(cam_names):
            if frame_idx in labeled_2d_poses and cam in labeled_2d_poses[frame_idx]:
                # Add left hand keypoints
                gt_kpts.append([list(kpts) + [0.0 if sum(kpts) == 0.0 else 2.0 if score > 0.5 else 1.0] for kpts, score in zip(labeled_2d_poses[frame_idx][cam][0]['keypoints'].squeeze(), labeled_2d_poses[frame_idx][cam][0]['scores'].squeeze())])
                pred_kpts.append([[kx, ky] + [0.0 if kx + ky == 0 else 2.0 if score > 0.5 else 1.0 if kx + ky > 0 else 0.0] for kx, ky, score in method_predictions[idx][cam_idx][:21]])

                # Add right hand keypoints
                gt_kpts.append([list(kpts) + [0.0 if sum(kpts) == 0.0 else 2.0 if score > 0.5 else 1.0] for kpts, score in zip(labeled_2d_poses[frame_idx][cam][1]['keypoints'].squeeze(), labeled_2d_poses[frame_idx][cam][1]['scores'].squeeze())])
                pred_kpts.append([[kx, ky] + [0.0 if kx + ky == 0 else 2.0 if score > 0.5 else 1.0 if kx + ky > 0 else 0.0] for kx, ky, score in method_predictions[idx][cam_idx][21:]])

    gt_kpts = np.array(gt_kpts)
    pred_kpts = np.array(pred_kpts)
    sigmas = np.array([.67] + [.25] * 20)
    areas = compute_areas(gt_kpts)

    # Compute OKS
    oks = np.zeros(len(gt_kpts))
    for i in range(len(gt_kpts)):
        oks_ = keypoint_similarity(gt_kpts[i], pred_kpts[i], sigmas, np.array([areas[i]]))
        oks[i] = oks_.item()

    # Compute average precision (AP) and average recall (AR)
    ap = compute_average_precision(oks)
    pck = compute_pck(gt_kpts, pred_kpts)
    mje = compute_mean_joint_error(gt_kpts, pred_kpts)

    return ap, pck, mje

evaluation_results = {}

for data_collection in data_collections:
    print(f"\nEvaluating data collection: {data_collection}")

    # Load ground truth for the current data collection
    labeled_2d_poses = load_labeled_2d_poses(data_collection, labeled_frames)

    # Evaluate each method
    for method_name, method_predictions in poses_2d[data_collection].items():
        if method_name not in evaluation_results:
            evaluation_results[method_name] = {}
        
        # Run evaluation
        ap, pck, mje = evaluate_keypoint_predictions(labeled_2d_poses, method_predictions)
        evaluation_results[method_name][data_collection] = {'AP': ap, 'PCK': pck, 'MJE': mje}

# Display results
print("\n=== Evaluation Results ===")
for method_name, results in evaluation_results.items():
    print(f"\n{method_name}:")
    print(f"  results: {results}")
    print(f"  mAP: {np.mean([v['AP'] for v in results.values()]) * 100:.3f}%")
    print(f"  mPCK: {np.mean([v['PCK'] for v in results.values()]) * 100:.3f}%")
    print(f"  mMJE: {np.mean([v['MJE'] for v in results.values()]):.3f}")



Evaluating data collection: cha/cha1

Evaluating data collection: cha/cha2

Evaluating data collection: cha/cha3

Evaluating data collection: cha/cha4

Evaluating data collection: cha/cha5

Evaluating data collection: cha/cha6

Evaluating data collection: cha/cha7

=== Evaluation Results ===

Sapiens:
  results: {'cha/cha1': {'AP': 0.5306249966835939, 'PCK': 0.1849702380952381, 'MJE': 29.781734751299158}, 'cha/cha2': {'AP': 0.5474999965781251, 'PCK': 0.1851934523809524, 'MJE': 94.06719002936026}, 'cha/cha3': {'AP': 0.5712499964296875, 'PCK': 0.19776785714285713, 'MJE': 35.01642912017243}, 'cha/cha4': {'AP': 0.5762499963984375, 'PCK': 0.17782738095238096, 'MJE': 34.38940765513428}, 'cha/cha5': {'AP': 0.7237499954765625, 'PCK': 0.10081845238095238, 'MJE': 39.925550001051484}, 'cha/cha6': {'AP': 0.7249999954687499, 'PCK': 0.09508928571428571, 'MJE': 36.57368380032853}, 'cha/cha7': {'AP': 0.6631249958554688, 'PCK': 0.11190476190476191, 'MJE': 33.455722839439076}}
  mAP: 61.964%
  mPCK: 15